In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os

import pandas as pd
import torch
import torch.nn as nn

from src.dataset import make_loaders
from src.evaluate import predict_loader, summarize
from src.model import CNNBaseline
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()

## Controlled ablations: 

In [ ]:
def save_result(row):
    new = pd.DataFrame([row])
    if os.path.exists(RESULTS):
        old = pd.read_csv(RESULTS)
        new = pd.concat([old[old['experiment'] != row['experiment']], new])
    new.to_csv(RESULTS, index=False)

def run_experiment(name, augment=False, dropout_p=0.0, pool_type='max',
                   weight_decay=0.0, scheduler_name='none', lr=1e-3, 
                   seed=SEED, epochs=20): 
    set_seed(seed)
    train_ds, val_ds, train_loader, val_loader = make_loaders(
        DATA, IMG_SIZE, BATCH, seed, augment)
    model = CNNBaseline(dropout_p=dropout_p, pool_type=pool_type).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler, step_on_val = make_scheduler(scheduler_name, optimizer)
    criterion = nn.CrossEntropyLoss()

    config = {
        'augment': augment, 'dropout_p': dropout_p, 'pool_type': pool_type,
        'weight_decay': weight_decay, 'scheduler': scheduler_name, 'lr': lr,
        'epochs': epochs,   
        'batch_size': BATCH, 'optimizer': 'AdamW'
    }
    meta = {
        'architecture': 'CNNBaseline',
        'class_to_idx': train_ds.class_to_idx,
        'img_size': IMG_SIZE, 'seed': seed, 'config': config
    }
    ckpt = f'../checkpoints/{name}.pt'
    history = fit(
        model, train_loader, val_loader, optimizer, criterion, device,
        epochs,  
        ckpt, f'../logs/{name}.csv',
        meta=meta, scheduler=scheduler, step_on_val=step_on_val
    )
    model.load_state_dict(torch.load(ckpt, weights_only=False)['model_state_dict'])
    trues, preds = predict_loader(model, val_loader, device)
    row = summarize(trues, preds, train_ds.classes)
    best = max(history, key=lambda r: r['val_macro_f1'])
    row.update({
        'experiment': name, 'best_epoch': best['epoch'],
        'train_loss': best['train_loss'], 'val_loss': best['val_loss'],
        'gap': best['val_loss'] - best['train_loss'],
        'final_lr': history[-1]['lr'], **config
    })
    save_result(row)
    return row

In [ ]:
run_experiment('baseline')

In [ ]:
run_experiment('aug' , augment= True)

In [ ]:
run_experiment('dropout_0.3', dropout_p=0.3)

In [ ]:
run_experiment('dropout_0.5', dropout_p=0.5)

In [ ]:
run_experiment('pool_avg', pool_type= 'avg')

{'epoch': 1, 'train_loss': 1.9568998077769337, 'val_loss': 1.8657154968806675, 'val_macro_f1': 0.2178182790582914, 'lr': 0.001}
{'epoch': 2, 'train_loss': 1.8193360097691795, 'val_loss': 1.8032104625330343, 'val_macro_f1': 0.25679010716526723, 'lr': 0.001}
{'epoch': 3, 'train_loss': 1.7598136966301297, 'val_loss': 1.799458748953683, 'val_macro_f1': 0.27944005028897567, 'lr': 0.001}
{'epoch': 4, 'train_loss': 1.7656848956579663, 'val_loss': 1.7306187251945595, 'val_macro_f1': 0.2981623168213314, 'lr': 0.001}
{'epoch': 5, 'train_loss': 1.712804292067837, 'val_loss': 1.7105976287420694, 'val_macro_f1': 0.30481831816730337, 'lr': 0.001}
{'epoch': 6, 'train_loss': 1.6946777766830663, 'val_loss': 1.7154762785156052, 'val_macro_f1': 0.3176296202933556, 'lr': 0.001}
